# exp_retriever_fusion — does a BM25 + retriever-v2 + BGE hybrid beat the current fusion?

The bake-off showed BGE is a much stronger *dense* component on 2023 (0.448 vs retriever-v2's 0.327),
but retriever-v2 wins in-domain (2021). A multi-retriever RRF hybrid should keep 2021 strong (retriever-v2
dominates there) and lift 2023 (BGE contributes) — an a-priori robustness choice, the retrieval analog of
the multi-view reranking. This measures RRF fusions by retrieval-order NDCG@10 on the same pools (cheap).

Current fusion = BM25 + retriever-v2 (TREC23 RRF = 0.472). We test adding/replacing with BGE.
Develop on TREC21, read across to TREC23. Requires the 2021 corpus + the cached 2023 files.

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q sentence-transformers rank-bm25 datasets pytrec_eval tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json
os.environ['HF_HUB_DISABLE_XET'] = '1'; os.environ['HF_HOME'] = '/content/hf_cache'
import numpy as np, torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from ctmatch.experiments import (ExperimentConfig, load_corpus, load_eval, build_bm25,
                                 retrieval_blob, full_blob, resolve_ckpt, rrf_fuse, pytrec_metrics)
DATA_ROOT = '/content/drive/MyDrive/ct_data23'; T23 = f'{DATA_ROOT}/trec2023'
cfg = ExperimentConfig(data_root=DATA_ROOT, pool_tag='nqs')

# ── TREC23 (test): pool + cached bm25/dense(retriever-v2) from retrieval_feats ──
id2f23 = {}
for l in open(f'{T23}/doc_fulltext_2023.jsonl'):
    r = json.loads(l); id2f23[r.get('nct_id') or r.get('doc_id')] = r
topics23 = {r['topic_id']: r['topic_text'] for r in map(json.loads, open(f'{T23}/topics2023_text.jsonl'))}
rel23 = {}
for l in open(f'{T23}/qrels2023.txt'):
    t, _, d, r = l.split(); rel23.setdefault(t, {})[d] = int(r)
topics23 = {t: x for t, x in topics23.items() if t in rel23}
pool23 = json.load(open(f'{T23}/pool_nqs_2023.json'))
bm25_23, v2_23 = {}, {}
for l in open(f'{T23}/retrieval_feats_2023.jsonl'):
    r = json.loads(l); bm25_23.setdefault(r['topic_id'], {})[r['doc_id']] = r['bm25']; v2_23.setdefault(r['topic_id'], {})[r['doc_id']] = r['dense']

# ── TREC21 (develop, judged pool): build BM25 + get retriever-v2/BGE by encoding ──
corpus_ids, corpus_fields = load_corpus(cfg); id2f21 = dict(zip(corpus_ids, corpus_fields))
s21 = load_eval(cfg, ['trec21'])['trec21']; rel21 = s21['rel_dict']; t2t21 = s21['topic2text']
topics21 = {t: t2t21[t] for t in rel21 if t in t2t21}
print('building TREC21 full-corpus BM25 (few min, CPU)...')
bm25_index = build_bm25(corpus_fields, cfg)
cid2row = {d: i for i, d in enumerate(corpus_ids)}
bm25_21 = {}
for t in tqdm(topics21, desc='bm25 trec21'):
    sc = bm25_index.get_scores(topics21[t].lower().split())
    bm25_21[t] = {d: float(sc[cid2row[d]]) for d in rel21[t] if d in cid2row}
print('TREC21 topics', len(topics21), '| TREC23 topics', len(topics23))

In [ ]:
# Encode retriever-v2 (TREC21 only — cached for 2023) and BGE (both) → per-topic per-doc cosine.
def st_scores(name, topics_d, docset_fn, id2f, q_prefix='', maxlen=512):
    m = SentenceTransformer(resolve_ckpt(cfg, name)); m.max_seq_length = maxlen
    tids = list(topics_d)
    qv = m.encode([q_prefix + topics_d[t] for t in tids], normalize_embeddings=True, batch_size=128, show_progress_bar=True).astype('float32')
    uniq = sorted({d for t in tids for d in docset_fn(t) if d in id2f})
    dv = m.encode([retrieval_blob(id2f[d], cfg) for d in uniq], normalize_embeddings=True, batch_size=128, show_progress_bar=True).astype('float32')
    didx = {d: i for i, d in enumerate(uniq)}
    out = {t: {d: float(qv[k] @ dv[didx[d]]) for d in docset_fn(t) if d in didx} for k, t in enumerate(tids)}
    del m; torch.cuda.empty_cache(); return out

BGE = 'BAAI/bge-large-en-v1.5'; BGEP = 'Represent this sentence for searching relevant passages: '
v2_21  = st_scores(cfg.retriever_ckpt, topics21, lambda t: list(rel21[t]), id2f21)
bge_21 = st_scores(BGE, topics21, lambda t: list(rel21[t]), id2f21, q_prefix=BGEP)
bge_23 = st_scores(BGE, topics23, lambda t: pool23[t], id2f23, q_prefix=BGEP)
print('encoded retriever-v2 (2021) + BGE (2021, 2023)')

In [ ]:
# RRF-fuse any set of per-doc score dicts, score retrieval order.
def fuse_score(score_dicts, topics_d, docset_fn, id2f, rel):
    run = {}
    for t in topics_d:
        docs = [d for d in docset_fn(t) if d in id2f]
        rankings = [sorted(docs, key=lambda d: sd[t].get(d, -1e9), reverse=True) for sd in score_dicts]
        run[t] = rrf_fuse(rankings, k=cfg.rrf_k)
    qrels = {t: {d: int(r) for d, r in rel[t].items()} for t in run}
    return pytrec_metrics(run, qrels, k=10)

COMBOS = {
    'BM25 + v2 (current)':   (['bm25', 'v2'],),
    'BM25 + BGE':            (['bm25', 'bge'],),
    'BM25 + v2 + BGE':       (['bm25', 'v2', 'bge'],),
    'v2 + BGE':              (['v2', 'bge'],),
}
SRC21 = {'bm25': bm25_21, 'v2': v2_21, 'bge': bge_21}
SRC23 = {'bm25': bm25_23, 'v2': v2_23, 'bge': bge_23}
print(f'{"fusion":24s} {"TREC21 NDCG@10":>15s} {"TREC23 NDCG@10":>15s}')
print('-'*58)
for name, (keys,) in COMBOS.items():
    m21 = fuse_score([SRC21[k] for k in keys], topics21, lambda t: list(rel21[t]), id2f21, rel21)
    m23 = fuse_score([SRC23[k] for k in keys], topics23, lambda t: pool23[t], id2f23, rel23)
    print(f'{name:24s} {m21["ndcg@10"]:>15.4f} {m23["ndcg@10"]:>15.4f}')
print('-'*58)
print('reference TREC23: current rrf 0.472 | ensemble 0.399 | IELAB dense 0.576 | oracle 1.000')
print('\nRead: does BM25+v2+BGE (or BM25+BGE) beat BM25+v2 on TREC23 without regressing TREC21?')
print('If yes -> full-corpus re-retrieval with that fusion (new pool -> re-extract features -> re-fit ensemble).')